# Data Parsing Pipeline


In [14]:
import pandas as pd
import json

# CONFIG
INPUT_XLSX = "../data/data_extraction.xlsx"
SHEET_NAME = "data_ordered"
OUTPUT_JSONL = "../data/parsed_extracted_data.jsonl"

DONE_VALUE = 1
EMPTY_COL_THRESHOLD = 0.15  # if <15% non-null → considered right-side table

# LOAD EXCEL
df = pd.read_excel(INPUT_XLSX, sheet_name=SHEET_NAME)

# AUTO-DETECT LEFT TABLE WIDTH
# (stop when columns become mostly empty)
non_null_ratio = df.notna().mean()
left_cols = non_null_ratio[non_null_ratio > EMPTY_COL_THRESHOLD].index.tolist()

left_df = df[left_cols].copy()

# BUILD HEADERS
# - col 0,1: Excel headers (Done ?, Who?)
# - others: headers stored in row 0
excel_headers = [str(c).strip() for c in left_df.columns.tolist()]
row0_headers = left_df.iloc[0].tolist()

headers = []
for i in range(len(left_df.columns)):
    if i < 2:
        headers.append(excel_headers[i])
    else:
        h = row0_headers[i]
        if pd.isna(h) or str(h).strip().lower() in {"nan", ""}:
            headers.append(excel_headers[i])
        else:
            headers.append(str(h).strip())

# DATA STARTS FROM ROW 1
data_df = left_df.iloc[1:].copy()

# FILTER ROWS
# 1. Keep processed rows (Done ? == 1)
# 2. Remove section rows (ID starts with "P_")
data_df = data_df[data_df.iloc[:, 0] == DONE_VALUE]

ID_COL_IDX = headers.index("ID")
data_df = data_df[~data_df.iloc[:, ID_COL_IDX].astype(str).str.startswith("P_")]

# DROP USELESS / EMPTY COLUMNS
clean_headers = []
clean_df = pd.DataFrame()

for i, h in enumerate(headers):
    col_series = data_df.iloc[:, i]

    # Drop if header is unnamed or empty
    if h is None or str(h).strip() == "" or str(h).lower().startswith("unnamed"):
        continue

    # Drop if column is entirely empty
    if col_series.notna().sum() == 0:
        continue

    clean_headers.append(h)
    clean_df[h] = col_series.values

# Replace with cleaned versions
headers = clean_headers
data_df = clean_df

# PARSE TO LIST OF DICTS
records = []
for _, row in clean_df.iterrows():
    record = {}
    for h, v in zip(headers, row.tolist()):
        if h and h.lower() != "nan":
            record[h] = None if pd.isna(v) else v
    records.append(record)


In [15]:
LIST_FIELDS = [
    "Authors",
    "Task",
    "Model",
    "Learning type",
    "Training method",
    "Input features",
    "Pre-processing techniques",
    "Encoding techniques",
    "Output",
    "Evaluation type",
    "Evaluation method",
    "Metrics",
    "Language",
]


In [16]:
def split_to_list(value):
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if not isinstance(value, str):
        return [value]

    return [v.strip() for v in value.replace(";", ",").split(",") if v.strip()]


for r in records:
    for field in LIST_FIELDS:
        if field in r:
            r[field] = split_to_list(r[field])


In [17]:
tasks_set = set()


In [18]:
for r in records:
    for t in r["Task"]:
        tasks_set.add(t)


In [19]:
tasks_set


{'Code refinement',
 'Code review data curation',
 'Code review gamification',
 'Comment Update',
 'Issue type prediction',
 'Issue/bug detection',
 'Named Entity Recognition',
 'Pull request completion time (effort estimation)',
 'Pull request prioritization',
 'Review comment evaluation',
 'Reviewer recommendation',
 'Sentiment analysis',
 'code compliance assessment',
 'pull request assessment',
 'review comment generation',
 'review comment recommendation'}

In [20]:
TASK_RENAME_MAP = {
    "Code refinement": "Code refinement",
    "Code review data curation": "Data curation",
    "Code review gamification": "Code review gamification",
    "Comment Update": "Comment generation",
    "Issue type prediction": "Issue type prediction",
    "Issue/bug detection": "Issue/bug detection",
    "Named Entity Recognition": "Comment evaluation",
    "Pull request completion time (effort estimation)": "Pull request assessment",
    "Pull request prioritization": "Pull request prioritization",
    "Review comment evaluation": "Comment evaluation",
    "Reviewer recommendation": "Reviewer recommendation",
    "Sentiment analysis": "Sentiment analysis",
    "code compliance assessment": "Issue/bug detection",
    "pull request assessment": "Pull request assessment",
    "review comment generation": "Comment generation",
    "review comment recommendation": "Comment recommendation",
}


In [21]:
for r in records:
    r["Task"] = [TASK_RENAME_MAP[t.strip()] for t in r["Task"]]


In [22]:
TASK_TYPE_MAP = {
    "Pull request assessment": "Pre-Review",
    "Reviewer recommendation": "Pre-Review",
    "Pull request prioritization": "Pre-Review",
    # Assisted
    "Issue/bug detection": "Assisted",
    "Issue type prediction": "Assisted",
    "Comment recommendation": "Assisted",
    # Generative
    "Comment generation": "Generative",
    "Code refinement": "Generative",
    "Data curation": "Generative",
    # Evaluation
    "Comment evaluation": "Evaluation",
    "Sentiment analysis": "Evaluation",
    "Code review gamification": "Evaluation",
}


In [23]:
for r in records:
    r["Task_Categories"] = list(set([TASK_TYPE_MAP[t] for t in r["Task"]]))


In [24]:
# SAVE JSONL
with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Saved {len(records)} records to {OUTPUT_JSONL}")
print("Last column parsed:", headers[-1])


Saved 165 records to ../data/parsed_extracted_data.jsonl
Last column parsed: Remarks


In [25]:
len(records)


165

In [26]:
records[12]


{'Done ?': 1.0,
 'Who?': 'Imen',
 'ID': 'P13',
 'Title': 'Early prediction of merged code changes to prioritize reviewing tasks',
 'Authors': ['Yuanrui Fan', 'Xin Xia', 'David Lo', 'Shanping Li'],
 'Venue type': 'Journal',
 'Venue': 'EMSE',
 'Year': 2018,
 'Task': ['Pull request assessment', 'Pull request prioritization'],
 'Automation level': 'semi-automated',
 'Model': ['RandomForest'],
 'Learning type': ['Classification'],
 'Model category': 'Machine Learning',
 'Model subcategory': 'Tree-Based / Ensemble Models',
 'Training method': ['Training'],
 'Input features': ['code diff',
  'PR metadata',
  'Reviewer metadata',
  'commit message',
  'Old code metadata'],
 'Pre-processing techniques': ['feature extraction'],
 'Encoding techniques': ['Binary encoding', 'Count Encoding'],
 'Output': ['merged/abonded', 'Ranked list of PRs'],
 'Dataset availability': 'not available',
 'Dataset': None,
 'Dataset size': 166215,
 'Language': ['Java', 'Python', 'C++'],
 'Evaluation type': ['Quantitat